# Lab Day 10: Finetune Qwen3-VL-4B bằng QLoRA trên Colab

Các cell code đã được điền theo các mục 0–8 của mẫu lab.
Chọn Colab T4, rồi chạy từ trên xuống. Có thể dùng ZIP hoặc Kaggle Secrets.
Mã bootstrap cung cấp công cụ đề; thuật toán huấn luyện và suy luận hiển thị trực tiếp trong cell.
Sao lưu adapter/projector mỗi 100 bước lên Drive nếu MOUNT_DRIVE=True.
Kết quả: `/content/lab_day10_results.zip`; số liệu được tạo bởi score.py gốc.
E0–E4 là phần bắt buộc đầy đủ; E5/GRPO vẫn là bonus chưa triển khai.


## 0. Môi trường (Colab T4) và cấu hình

In [ ]:
# Colab: Runtime → Change runtime type → T4 GPU.
# ZIP: upload archive.zip vào /content, rồi đặt DATA_ZIP="/content/archive.zip".
# Hoặc để DATA_ZIP=None, RAW_DATA_DIR=None và dùng Colab Secrets cho Kaggle CLI.
DATA_ZIP = None
RAW_DATA_DIR = None  # thư mục gốc có images/ và labels-YOLO/, nếu đã giải nén
MOUNT_DRIVE = True  # đặt False nếu chỉ lưu trong session hiện tại
DRIVE_BACKUP_DIR = "/content/drive/MyDrive/lab_day10_backup"
RESUME_SOURCE = None  # có thể trỏ vào DRIVE_BACKUP_DIR của lượt hoàn thành trước
MODEL_REVISION = None  # khi resume, dùng SHA trong kaggle_run_config.json cũ
MAX_STEPS = 300
BATCH_SIZE = 1
GRAD_ACCUM = 8
INFERENCE_BATCH_SIZE = 2
MAX_NEW_TOKENS = 512
DOWNLOAD_RESULTS = True


### Khôi phục công cụ đề và mã nguồn dùng cho kiểm thử
Cell bootstrap tạo scripts/tests/configs và các file Python đối chiếu. Phần thực nghiệm được thực thi bằng các cell bên dưới.

In [ ]:
import base64, io, json, os, subprocess, sys, zipfile
from pathlib import Path

BASE = Path("/content")
BASE.mkdir(parents=True, exist_ok=True)
PROJECT = BASE / "day10"
PROJECT.mkdir(exist_ok=True)
PAYLOAD = "..."
with zipfile.ZipFile(io.BytesIO(base64.b64decode(PAYLOAD))) as archive:
    archive.extractall(PROJECT)
os.environ["HF_HOME"] = "/content/day10_hf_cache"  # model gốc không nằm trong Output
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONUNBUFFERED"] = "1"
config = {"input_root": "/content", "data_source": RAW_DATA_DIR,
          "resume_source": RESUME_SOURCE, "model_revision": MODEL_REVISION,
          "max_steps": MAX_STEPS, "batch_size": BATCH_SIZE, "grad_accum": GRAD_ACCUM,
          "inference_batch_size": INFERENCE_BATCH_SIZE, "max_new_tokens": MAX_NEW_TOKENS}
if BATCH_SIZE * GRAD_ACCUM != 8:
    raise ValueError("Giữ effective batch size = 8 cho các thí nghiệm.")
if not all(isinstance(v, int) and v > 0 for v in
           [MAX_STEPS, BATCH_SIZE, GRAD_ACCUM, INFERENCE_BATCH_SIZE, MAX_NEW_TOKENS]):
    raise ValueError("Các cấu hình bước/batch/token phải là số nguyên dương.")
(PROJECT / "kaggle_run_config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")
print("Project:", PROJECT)


In [ ]:
# Cài trước khi kernel import torch/transformers. Giữ bản CUDA torch/vision của Colab.
subprocess.run(["nvidia-smi", "-L"], check=True)
probe = subprocess.run([sys.executable, "-c",
    "import torch, torchvision; assert torch.cuda.is_available(), 'GPU required'; "
    "print('torch',torch.__version__,'torchvision',torchvision.__version__); "
    "assert tuple(map(int,torch.__version__.split('+')[0].split('.')[:2])) >= (2,4), 'torch>=2.4 required'"],
    check=True)
import importlib.metadata as package_metadata
constraints = PROJECT / "colab_torch_constraints.txt"
versions = []
for name in ("torch", "torchvision", "torchaudio"):
    try:
        versions.append(name + "==" + package_metadata.version(name))
    except package_metadata.PackageNotFoundError:
        pass
constraints.write_text(chr(10).join(versions) + chr(10), encoding="utf-8")
print("Keeping Colab CUDA packages:", versions)
subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
                "-r", str(PROJECT / "requirements-kaggle.txt"), "kaggle", "-c", str(constraints)], check=True)
# This pipeline uses bitsandbytes NF4. A preinstalled TorchAO can break PEFT's
# optional backend detection (PEFT 0.21.2 rejects torchao < 0.16.0).
try:
    torchao_version = package_metadata.version("torchao")
except package_metadata.PackageNotFoundError:
    torchao_version = None
if torchao_version is not None:
    print("Removing unused optional TorchAO backend:", torchao_version)
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests", "kaggle_checks"],
               cwd=PROJECT, check=True)


In [ ]:
import gc, hashlib, importlib.metadata, json, os, platform
import random, re, shutil, subprocess, sys, time, traceback, zipfile
from pathlib import Path
import torch
from IPython.display import display, FileLink

ROOT = PROJECT
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from scripts.labcommon import read_jsonl
base = json.loads((ROOT / "kaggle_run_config.json").read_text(encoding="utf-8"))


### Mount Google Drive để lưu checkpoint và kết quả

In [ ]:
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BACKUP_ROOT = Path(DRIVE_BACKUP_DIR)
    BACKUP_ROOT.mkdir(parents=True, exist_ok=True)
else:
    BACKUP_ROOT = None
print("Checkpoint backup:", BACKUP_ROOT)


def persist_outputs():
    """Backup logs, predictions, checkpoint and results; exclude raw dataset/cache."""
    if BACKUP_ROOT is None:
        return
    work = ROOT / "work"
    if work.exists():
        (BACKUP_ROOT / "work").mkdir(parents=True, exist_ok=True)
        for item in work.iterdir():
            if item.name in ("raw", "dataset.zip"):
                continue
            destination = BACKUP_ROOT / "work" / item.name
            if item.is_dir():
                shutil.copytree(item, destination, dirs_exist_ok=True)
            else:
                shutil.copy2(item, destination)
    if (ROOT / "report").exists():
        shutil.copytree(ROOT / "report", BACKUP_ROOT / "report", dirs_exist_ok=True)
    shutil.copy2(ROOT / "kaggle_run_config.json", BACKUP_ROOT / "kaggle_run_config.json")
    archive = ROOT.parent / "lab_day10_results.zip"
    if archive.exists():
        shutil.copy2(archive, BACKUP_ROOT / archive.name)
    print("Saved to Drive:", BACKUP_ROOT)


In [ ]:
def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")
    temp.replace(path)

def assert_disjoint_sessions(splits):
    sessions = {s: {r["session"] for r in rows} for s, rows in splits.items()}
    names = list(sessions)
    for i, a in enumerate(names):
        for b in names[i + 1:]:
            if sessions[a] & sessions[b]:
                raise ValueError(f"Session leakage between {a} and {b}")
    return {s: len(v) for s, v in sessions.items()}

def command(args, log_path=None):
    args = [str(v) for v in args]
    if log_path is None:
        subprocess.run(args, cwd=ROOT, check=True)
        return
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(args, cwd=ROOT, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, encoding="utf-8",
                                   errors="replace", bufsize=1)
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
        if process.wait():
            raise subprocess.CalledProcessError(process.returncode, args)

def image_path(row):
    return ROOT / "data" / row["image_path"]

def find_dataset(input_root, explicit):
    if explicit:
        path = Path(explicit)
        if path.is_file() and path.suffix.lower() == ".zip":
            return "--zip", path
        if (path / "images").is_dir() and (path / "labels-YOLO").is_dir():
            return "--source", path
        raise FileNotFoundError(f"Invalid DATA_SOURCE: {path}")
    roots = sorted({p.parent for p in Path(input_root).rglob("labels-YOLO")
                    if p.is_dir() and (p.parent / "images").is_dir()
                    and "work" not in p.relative_to(input_root).parts})
    if len(roots) == 1:
        return "--source", roots[0]
    if len(roots) > 1:
        raise RuntimeError("Multiple datasets found; set DATA_SOURCE explicitly.")
    zips = []
    for p in Path(input_root).rglob("*.zip"):
        try:
            with zipfile.ZipFile(p) as z:
                if any("labels-YOLO/" in n for n in z.namelist()):
                    zips.append(p)
        except zipfile.BadZipFile:
            continue
    if len(zips) == 1:
        return "--zip", zips[0]
    raise FileNotFoundError("Add the road-damage dataset in Kaggle Input, or set DATA_SOURCE.")


## 1. Dữ liệu: tải và chia split

In [ ]:
(ROOT / "work").mkdir(exist_ok=True)
(ROOT / "report").mkdir(exist_ok=True)
if DATA_ZIP is not None and RAW_DATA_DIR is not None:
    raise ValueError("Chọn đúng một nguồn: DATA_ZIP hoặc RAW_DATA_DIR.")
if RAW_DATA_DIR is not None:
    command([sys.executable, "scripts/prepare_data.py", "--source", RAW_DATA_DIR])
elif DATA_ZIP is not None:
    if not Path(DATA_ZIP).is_file():
        raise FileNotFoundError("Upload ZIP trước khi chạy, hoặc để DATA_ZIP=None để dùng Kaggle Secrets.")
    command([sys.executable, "scripts/prepare_data.py", "--zip", DATA_ZIP])
else:
    from google.colab import userdata
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    command([sys.executable, "scripts/prepare_data.py", "--kaggle"])
splits = {s: read_jsonl(ROOT / f"data/splits/{s}.jsonl") for s in ("train", "val", "test")}
dataset_meta = json.loads((ROOT / "data/splits/meta.json").read_text(encoding="utf-8"))
base["dataset_version"] = dataset_meta["dataset_version"]
session_counts = assert_disjoint_sessions(splits)
for split, rows in splits.items():
    print(split, "images=", len(rows), "sessions=", session_counts[split],
          "boxes=", dataset_meta["counts"][split]["boxes"])
print("dataset_version:", base["dataset_version"])


In [ ]:
# Vẽ 6 ảnh với nhãn YOLO gốc, trước khi chuyển sang JSON.
from PIL import Image, ImageDraw
for row in splits["train"][:6]:
    with Image.open(image_path(row)) as original:
        image = original.convert("RGB")
    draw = ImageDraw.Draw(image)
    width, height = image.size
    for obj in row["objects"]:
        xc, yc, w, h = obj["yolo"]
        box = [(xc-w/2)*width, (yc-h/2)*height, (xc+w/2)*width, (yc+h/2)*height]
        draw.rectangle(box, outline="lime", width=2)
        draw.text((box[0], box[1]), obj["label"], fill="lime")
    display(image)


## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
"""CPU-only data and artifact helpers for the student Kaggle implementation."""
import csv
import hashlib
import json
import random
import re
from pathlib import Path

LLM_TARGET = r"model\.language_model\.layers\.\d+\.(?:self_attn\.(?:q|k|v|o)_proj|mlp\.(?:gate|up|down)_proj)"
VIT_TARGET = r"model\.visual\.blocks\.\d+\.(?:attn\.(?:qkv|proj)|mlp\.(?:linear_fc1|linear_fc2))"
PROJECTORS = ("visual.merger", "visual.deepstack_merger_list")
PROMPT = (
    'Detect all road damage: pothole, crack, and manhole. Return ONLY a JSON list '
    'of objects with keys "label" and "bbox_2d". bbox_2d is [x1,y1,x2,y2], '
    'integer coordinates normalized to 0-1000 independently for image width and height. '
    'Use only these three labels. Return [] if there are no objects. No explanations.'
)


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")
    temp.replace(path)


def yolo_to_box(label, yolo):
    xc, yc, w, h = yolo
    coords = (xc - w / 2, yc - h / 2, xc + w / 2, yc + h / 2)
    box = [max(0, min(1000, round(v * 1000))) for v in coords]
    if box[2] <= box[0] or box[3] <= box[1]:
        return None
    return {"label": label, "bbox_2d": box}


def objects_to_answer(objects):
    boxes = [box for o in objects if (box := yolo_to_box(o["label"], o["yolo"])) is not None]
    boxes.sort(key=lambda o: (o["bbox_2d"][1], o["bbox_2d"][0]))
    return json.dumps(boxes, separators=(",", ":"))


def subset_train(records, fraction, seed):
    if not 0 < fraction <= 1:
        raise ValueError("train_fraction must be in (0,1]")
    if fraction == 1:
        return list(records)
    indices = random.Random(seed).sample(range(len(records)), max(1, int(len(records) * fraction)))
    return [records[i] for i in sorted(indices)]


def assert_disjoint_sessions(splits):
    sessions = {s: {r["session"] for r in rows} for s, rows in splits.items()}
    names = list(sessions)
    for i, a in enumerate(names):
        for b in names[i + 1:]:
            if sessions[a] & sessions[b]:
                raise ValueError(f"Session leakage between {a} and {b}")
    return {s: len(v) for s, v in sessions.items()}


def target_names(names, include_vit=False):
    regex = LLM_TARGET if not include_vit else f"(?:{LLM_TARGET}|{VIT_TARGET})"
    return [n for n in names if re.fullmatch(regex, n)]


def fingerprint(config):
    return hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()


def summary_rows(evidence_dir):
    rows = []
    for p in sorted(Path(evidence_dir).glob("*.json")):
        d = json.loads(p.read_text(encoding="utf-8"))
        rows.append({
            "experiment": p.stem, "split": d["split"], "f1": d["micro"]["f1"],
            "ci95_low": d["f1_ci95"][0], "ci95_high": d["f1_ci95"][1],
            "fp_per_image": d["fp_per_image"], "parse_valid_rate": d["parse_valid_rate"],
            "missing_predictions": d["missing_predictions"],
            **{f"f1_{c}": d["per_class"][c]["f1"] for c in ("pothole", "crack", "manhole")},
            **{k: d.get("meta", {}).get(k, 0) for k in (
                "trainable_params", "peak_vram_gb", "train_minutes")},
        })
    return rows


def write_summary(evidence_dir, output):
    rows = summary_rows(evidence_dir)
    if rows:
        with Path(output).open("w", encoding="utf-8", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=list(rows[0]))
            writer.writeheader()
            writer.writerows(rows)
    return rows


In [ ]:
def draw_boxes(row, pred, output, caption):
    from PIL import Image, ImageDraw
    with Image.open(image_path(row)) as image:
        image = image.convert("RGB")
    w, h = image.size
    canvas = Image.new("RGB", (w, h + 65), "white")
    canvas.paste(image, (0, 0))
    draw = ImageDraw.Draw(canvas)
    for o in row["objects"]:
        converted = yolo_to_box(o["label"], o["yolo"])
        if converted is None:
            continue
        bb = converted["bbox_2d"]
        box = [bb[0] * w / 1000, bb[1] * h / 1000, bb[2] * w / 1000, bb[3] * h / 1000]
        draw.rectangle(box, outline="lime", width=2)
        draw.text((box[0], max(0, box[1] - 12)), "GT:" + o["label"], fill="lime")
    for o in pred or []:
        bb = o["bbox"]
        box = [max(0, min(w, bb[0] * w / 1000)), max(0, min(h, bb[1] * h / 1000)),
               max(0, min(w, bb[2] * w / 1000)), max(0, min(h, bb[3] * h / 1000))]
        if box[2] >= box[0] and box[3] >= box[1]:
            draw.rectangle(box, outline="red", width=2)
            draw.text((box[0], max(0, box[1])), "P:" + o["label"], fill="red")
    draw.text((5, h + 4), caption[:100], fill="black")
    draw.text((5, h + 22), "GT=green | prediction=red | coordinates: 0-1000", fill="black")
    draw.text((5, h + 40), row["image_id"], fill="black")
    output.parent.mkdir(parents=True, exist_ok=True)
    canvas.save(output)


In [ ]:
# Bỏ nhãn train suy biến khỏi đáp án; giữ nguyên split và GT của đề.
degenerate = [{"split": s, "image_id": row["image_id"], "object": obj}
              for s, rows in splits.items() for row in rows for obj in row["objects"]
              if yolo_to_box(obj["label"], obj["yolo"]) is None]
if any(item["split"] != "train" for item in degenerate):
    raise RuntimeError("Có nhãn val/test suy biến; kiểm tra dataset trước khi chấm.")
write_json(ROOT / "report/data_checks.json", {
    "sessions": session_counts, "meta": dataset_meta,
    "degenerate_training_objects": degenerate,
    "policy": "Skip degenerate objects in training answers only; original splits/GT unchanged."})
print("Prompt:", PROMPT)
print("Example answer:", objects_to_answer(splits["train"][0]["objects"]))
print("Degenerate training objects:", degenerate)
for row in splits["train"][:6]:
    path = ROOT / "report/figures/data" / f"{row['image_id']}.png"
    draw_boxes(row, [], path, "Training label check (normalized round-trip)")
display(Image.open(ROOT / "report/figures/data" / f"{splits['train'][0]['image_id']}.png"))


## 3. E0: zero-shot

In [ ]:
def user_message(row):
    return {"role": "user", "content": [
        {"type": "image", "image": str(image_path(row))},
        {"type": "text", "text": PROMPT},
    ]}

def make_quantization_config():
    import torch
    from transformers import BitsAndBytesConfig
    # Transformers 5.19 matches exclusions against full names from named_modules().
    # "visual" does NOT exclude children named "model.visual.blocks...".
    return BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                              bnb_4bit_compute_dtype=torch.float16,
                              bnb_4bit_use_double_quant=True,
                              llm_int8_skip_modules=["model.visual", "lm_head"])

def load_base(cfg):
    import torch
    import bitsandbytes as bnb
    from transformers import AutoProcessor, Qwen3VLForConditionalGeneration
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA GPU is required. Enable a Kaggle GPU accelerator.")
    quant = make_quantization_config()
    model = Qwen3VLForConditionalGeneration.from_pretrained(
        cfg["model_id"], revision=cfg["model_revision"], quantization_config=quant,
        dtype=torch.float16, device_map={"": 0}, attn_implementation="sdpa")
    processor = AutoProcessor.from_pretrained(cfg["model_id"], revision=cfg["model_revision"])
    processor.tokenizer.padding_side = "left"
    if processor.tokenizer.pad_token_id is None:
        processor.tokenizer.pad_token = processor.tokenizer.eos_token
    bad = [n for n, m in model.named_modules()
           if ("visual" in n or n == "lm_head") and isinstance(m, bnb.nn.Linear4bit)]
    if bad:
        raise RuntimeError(f"Visual/lm_head unexpectedly quantized: {bad[:5]}")
    quantized_names = [n for n, m in model.named_modules() if isinstance(m, bnb.nn.Linear4bit)]
    if not quantized_names:
        raise RuntimeError("No NF4 layers: QLoRA was not activated.")
    print(f"NF4 layers: {len(quantized_names)}; visual and lm_head excluded from quantization", flush=True)
    return model, processor

def architecture(model):
    import bitsandbytes as bnb
    counts = {"llm_including_embeddings": 0, "vit_excluding_projector": 0,
              "projector": 0, "lm_head": 0, "other": 0}
    # Params4bit.numel() is packed; quant_state.shape holds the original dense shape.
    for n, p in model.named_parameters():
        if any(s in n for s in PROJECTORS):
            key = "projector"
        elif "visual" in n:
            key = "vit_excluding_projector"
        elif "language_model" in n:
            key = "llm_including_embeddings"
        elif "lm_head" in n:
            key = "lm_head"
        else:
            key = "other"
        if isinstance(p, bnb.nn.Params4bit) and p.quant_state is not None:
            import math
            size = math.prod(p.quant_state.shape)
        else:
            size = p.numel()
        counts[key] += size
    counts["total_dense_equivalent"] = sum(counts.values())
    return counts

def seed_everything(seed):
    import numpy as np
    import torch
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.benchmark = False


In [ ]:
def predict(model, processor, rows, output, cfg):
    import torch
    model.eval()
    model.config.use_cache = True
    batch_size = cfg["inference_batch_size"]
    index = 0
    start = time.perf_counter()
    tmp = output.with_suffix(".jsonl.tmp")
    with tmp.open("w", encoding="utf-8") as f:
        while index < len(rows):
            batch_rows = rows[index:index + batch_size]
            try:
                # A list of conversations, one image per conversation.
                inputs = processor.apply_chat_template(
                    [[user_message(r)] for r in batch_rows], tokenize=True,
                    add_generation_prompt=True, return_dict=True, return_tensors="pt",
                    processor_kwargs={"padding": True})
                inputs.pop("token_type_ids", None)
                inputs = {k: v.to("cuda:0") for k, v in inputs.items()}
                with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):
                    out = model.generate(**inputs, max_new_tokens=cfg["max_new_tokens"],
                                         do_sample=False, use_cache=True,
                                         pad_token_id=processor.tokenizer.pad_token_id)
                raw = processor.batch_decode(out[:, inputs["input_ids"].shape[1]:],
                                             skip_special_tokens=True,
                                             clean_up_tokenization_spaces=False)
                lengths = [int((tokens != processor.tokenizer.pad_token_id).sum())
                           for tokens in out[:, inputs["input_ids"].shape[1]:]]
                del inputs, out
            except torch.cuda.OutOfMemoryError:
                # OOM can leave partial tensors alive; release them before retrying.
                inputs = out = None
                gc.collect()
                torch.cuda.empty_cache()
                if batch_size == 1:
                    raise
                batch_size = max(1, batch_size // 2)
                print(f"Inference OOM: retry with batch_size={batch_size}", flush=True)
                continue
            for row, text, length in zip(batch_rows, raw, lengths):
                f.write(json.dumps({"image_id": row["image_id"], "raw": text,
                                    "generated_tokens": length,
                                    "hit_token_limit": length >= cfg["max_new_tokens"]}) + "\n")
            f.flush()
            index += len(batch_rows)
            if index % 20 < batch_size or index == len(rows):
                print(f"{output.stem}: {index}/{len(rows)}", flush=True)
    tmp.replace(output)
    return {"minutes": (time.perf_counter() - start) / 60, "batch_size_final": batch_size}

def score(name, split, meta):
    suffix = "" if split == "test" else "_val"
    dest = ROOT / "report" / ("evidence" if split == "test" else "validation") / f"{name}.json"
    args = [sys.executable, "scripts/score.py", "--pred", f"work/{name}{suffix}.jsonl",
            "--split", split, "--name", name.upper(), "--out", dest]
    for k, v in meta.items():
        args += ["--meta", f"{k}={v}"]
    command(args)
    return json.loads(dest.read_text(encoding="utf-8"))

def make_examples(name, rows):
    from scripts.score import parse_output, _gt_boxes, _match
    predictions = {r["image_id"]: r for r in read_jsonl(ROOT / "work" / f"{name}.jsonl")}
    details = []
    for row in rows:
        raw = predictions[row["image_id"]]["raw"]
        pred = parse_output(raw)
        per, _ = _match(_gt_boxes(row), pred or [])
        tp, fp, fn = [sum(v[k] for v in per.values()) for k in range(3)]
        details.append({"image_id": row["image_id"], "tp": tp, "fp": fp, "fn": fn,
                        "parse_valid": pred is not None, "raw": raw,
                        "hit_token_limit": predictions[row["image_id"]]["hit_token_limit"]})
    write_json(ROOT / "report" / "errors" / f"{name}.json", details)
    good = sorted(details, key=lambda d: (d["fp"] + d["fn"], -d["tp"], d["image_id"]))[:3]
    good_ids = {d["image_id"] for d in good}
    bad = sorted((d for d in details if d["image_id"] not in good_ids),
                 key=lambda d: (-(d["fp"] + d["fn"]), d["image_id"]))[:3]
    by_id = {r["image_id"]: r for r in rows}
    captions = []
    for i, d in enumerate(good + bad):
        caption = f"{name.upper()} TP={d['tp']} FP={d['fp']} FN={d['fn']} valid={d['parse_valid']}"
        path = ROOT / "report" / "figures" / name / f"{i + 1}_{d['image_id']}.png"
        draw_boxes(by_id[d["image_id"]], parse_output(d["raw"]), path, caption)
        captions.append({**d, "figure": str(path.relative_to(ROOT)), "caption": caption})
    write_json(ROOT / "report" / "figures" / name / "captions.json", captions)


In [ ]:
def config_for(base, name, setup=None, fraction=1.0):
    import yaml
    setup = setup or name
    if setup == "e0":
        original = yaml.safe_load((ROOT / "configs/e0.yaml").read_text(encoding="utf-8"))
        original.update(lr=0, seed=42, max_steps=0)
    else:
        original = yaml.safe_load((ROOT / f"configs/{setup}.yaml").read_text(encoding="utf-8"))
    return {"name": name, "setup": setup, "model_id": original["model_id"],
            "model_revision": base["model_revision"], "seed": original["seed"],
            "rank": original.get("lora", {}).get("r", 16), "lr": float(original["lr"]),
            "max_steps": base["max_steps"] if setup != "e0" else 0,
            "batch_size": base["batch_size"], "grad_accum": base["grad_accum"],
            "max_new_tokens": base["max_new_tokens"],
            "inference_batch_size": base["inference_batch_size"],
            "train_fraction": fraction, "prompt": PROMPT,
            "dataset_version": base["dataset_version"], "code_sha256": base["code_sha256"]}

def restore_completed(base, cfg):
    source = base.get("resume_source")
    if not source:
        return False
    source = Path(source)
    name = cfg["name"]
    marker = source / "work" / name / "completed.json"
    if not marker.exists():
        return False
    done = json.loads(marker.read_text(encoding="utf-8"))
    if done.get("fingerprint") != fingerprint(cfg):
        raise RuntimeError(f"Resume config mismatch for {name}; clear RESUME_SOURCE or use identical config.")
    required = [source / "report/evidence" / f"{name}.json",
                source / "report/validation" / f"{name}.json",
                source / "work" / f"{name}.jsonl", source / "work" / f"{name}_val.jsonl"]
    if not all(p.exists() for p in required):
        raise RuntimeError(f"Incomplete completed-run artifacts for {name}")
    shutil.copytree(source / "work" / name, ROOT / "work" / name, dirs_exist_ok=True)
    for p in required:
        dest = ROOT / p.relative_to(source)
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(p, dest)
    for part in (f"report/figures/{name}",):
        if (source / part).exists():
            shutil.copytree(source / part, ROOT / part, dirs_exist_ok=True)
    if (source / f"report/errors/{name}.json").exists():
        dest = ROOT / f"report/errors/{name}.json"
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source / f"report/errors/{name}.json", dest)
    print(f"Reused completed {name} from previous output", flush=True)
    return True

def bundle_results():
    rows = write_summary(ROOT / "report/evidence", ROOT / "report/summary.csv")
    manifest = {"evidence": rows, "report_written": False,
                "notes": ["Measured results only; REPORT.md must be completed after analysis.",
                          "Image-bootstrap CI is not a paired significance test or session-bootstrap CI."]}
    if (ROOT / "report/failure.json").exists():
        manifest["failure"] = json.loads((ROOT / "report/failure.json").read_text(encoding="utf-8"))
    write_json(ROOT / "report/results_manifest.json", manifest)
    paths = [*sorted((ROOT / "report").rglob("*")),
             *sorted((ROOT / "data/splits").glob("*.json*")),
             *sorted((ROOT / "work").rglob("*.json")),
             *sorted((ROOT / "work").rglob("*.jsonl")),
             *sorted((ROOT / "work").rglob("*.txt"))]
    # Exclude model weights, optimizer states, dataset images and HF model cache.
    excluded = {"tokenizer.json", "tokenizer_config.json", "special_tokens_map.json",
                "added_tokens.json", "vocab.json", "merges.txt", "chat_template.json"}
    dest = ROOT.parent / "lab_day10_results.zip"
    with zipfile.ZipFile(dest, "w", zipfile.ZIP_DEFLATED) as z:
        for p in paths:
            if p.is_file() and "adapter" not in p.relative_to(ROOT).parts and p.name not in excluded:
                z.write(p, p.relative_to(ROOT))
        for p in [ROOT / "kaggle_run_config.json", ROOT / "kaggle_lab.py", ROOT / "kaggle_utils.py"]:
            if p.exists():
                z.write(p, p.name)
    print(f"Results for analysis: {dest}", flush=True)


In [ ]:
# Cùng một revision model cho mọi thí nghiệm độc lập.
from huggingface_hub import model_info
base["model_revision"] = base.get("model_revision") or model_info("Qwen/Qwen3-VL-4B-Instruct").sha
base["code_sha256"] = hashlib.sha256(
    (ROOT / "kaggle_lab.py").read_bytes() + (ROOT / "kaggle_utils.py").read_bytes()).hexdigest()
base["execution_style"] = "colab_notebook_cells"
write_json(ROOT / "kaggle_run_config.json", base)
environment = {"python": sys.version, "platform": platform.platform(), "packages": {}}
for name in ("torch", "torchvision", "transformers", "peft", "bitsandbytes", "accelerate"):
    environment["packages"][name] = importlib.metadata.version(name)
environment["nvidia_smi"] = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
write_json(ROOT / "report/environment.json", environment)
print("Model revision:", base["model_revision"])


In [ ]:
def evaluate_setup(cfg):
    """Đánh giá một setup; luôn giải phóng GPU và đóng gói kết quả khi kết thúc cell."""
    name = cfg["name"]
    run_dir = ROOT / "work" / name
    run_dir.mkdir(parents=True, exist_ok=True)
    model = processor = None
    try:
        if restore_completed(base, cfg):
            make_examples(name, splits["test"])
            return
        seed_everything(cfg["seed"])
        write_json(run_dir / "effective_config.json", cfg)
        if name != "e0":
            run = json.loads((run_dir / "run.json").read_text(encoding="utf-8"))
            if fingerprint(run["config"]) != fingerprint(cfg) or run["optimizer_steps"] != cfg["max_steps"]:
                raise RuntimeError("Checkpoint/config không khớp lượt train đầy đủ.")
            meta = {k: run[k] for k in ("trainable_params", "peak_vram_gb", "train_minutes")}
        else:
            meta = {"trainable_params": 0, "peak_vram_gb": 0, "train_minutes": 0}
        model, processor = load_base(cfg)
        write_json(run_dir / "architecture.json", architecture(model))
        (run_dir / "module_names.txt").write_text(
            "\n".join(n for n, _ in model.named_modules()), encoding="utf-8")
        if name != "e0":
            model = reload_weights(model, run_dir, cfg)
        else:
            torch.cuda.reset_peak_memory_stats()
        inference = {}
        for split in ("val", "test"):
            suffix = "" if split == "test" else "_val"
            inference[split] = predict(model, processor, splits[split],
                                      ROOT / "work" / f"{name}{suffix}.jsonl", cfg)
            if name == "e0":
                meta["peak_vram_gb"] = torch.cuda.max_memory_allocated() / 2**30
            score(name, split, meta)
        write_json(run_dir / "inference.json", inference)
        if name == "e0":
            write_json(run_dir / "run.json", {**meta, "config": cfg, "status": "zero_shot",
                "inference_minutes": sum(v["minutes"] for v in inference.values()),
                "vram_unit": "GiB (2**30 bytes)"})
        make_examples(name, splits["test"])
        write_json(run_dir / "completed.json", {"fingerprint": fingerprint(cfg), "config": cfg})
    except Exception as exc:
        write_json(ROOT / "report/failure.json", {
            "phase": "evaluate", "experiment": name, "error": str(exc), "traceback": traceback.format_exc()})
        raise
    finally:
        model = processor = None
        gc.collect()
        torch.cuda.empty_cache()
        bundle_results()
        persist_outputs()



In [ ]:
cfg_e0 = config_for(base, "e0")
evaluate_setup(cfg_e0)
print("E0 test:", json.loads((ROOT / "report/evidence/e0.json").read_text())["micro"])


## 4. Dữ liệu huấn luyện: chat template và mask loss

In [ ]:
def encode(row, processor):
    import torch
    msg = user_message(row)
    prefix = processor.apply_chat_template(
        [msg], tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt")
    answer = objects_to_answer(row["objects"])
    full = processor.apply_chat_template(
        [msg, {"role": "assistant", "content": [{"type": "text", "text": answer}]}],
        tokenize=True, add_generation_prompt=False, return_dict=True, return_tensors="pt")
    n = prefix["input_ids"].shape[1]
    if not torch.equal(prefix["input_ids"], full["input_ids"][:, :n]):
        raise RuntimeError("Chat template prefix mismatch: cannot safely mask assistant loss.")
    keep = {"input_ids", "attention_mask", "mm_token_type_ids", "pixel_values", "image_grid_thw"}
    encoded = {k: v for k, v in full.items() if k in keep}
    encoded["labels"] = full["input_ids"].clone()
    encoded["labels"][:, :n] = -100
    if not (encoded["labels"] != -100).any():
        raise RuntimeError("No supervised answer tokens.")
    encoded["pixel_values"] = encoded["pixel_values"].half()
    return encoded, answer, n

def collate(batch, pad_id):
    import torch
    from torch.nn.functional import pad
    length = max(x["input_ids"].shape[1] for x in batch)
    out = {}
    for key, fill in (("input_ids", pad_id), ("attention_mask", 0),
                      ("mm_token_type_ids", 0), ("labels", -100)):
        if not all(key in x for x in batch):
            if key == "mm_token_type_ids":
                continue
            raise ValueError(f"Missing {key}")
        out[key] = torch.cat([pad(x[key], (0, length - x[key].shape[1]), value=fill)
                              for x in batch], dim=0)
    for key in ("pixel_values", "image_grid_thw"):
        out[key] = torch.cat([x[key] for x in batch], dim=0)
    return out


In [ ]:
from transformers import AutoProcessor
processor_check = AutoProcessor.from_pretrained("Qwen/Qwen3-VL-4B-Instruct", revision=base["model_revision"])
example, answer, prefix_length = encode(splits["train"][0], processor_check)
supervised = example["labels"][example["labels"] != -100]
decoded = processor_check.tokenizer.decode(supervised, skip_special_tokens=False)
image_id = processor_check.tokenizer.convert_tokens_to_ids("<|image_pad|>")
vision_tokens = int((example["input_ids"] == image_id).sum())
assert vision_tokens == 220
assert decoded.startswith(answer) and "<|im_end|>" in decoded
write_json(ROOT / "report/mask_example.json", {
    "vision_tokens": vision_tokens, "masked_prefix_tokens": prefix_length,
    "supervised_tokens": int(supervised.numel()), "decoded_supervised": decoded})
print("Vision tokens:", vision_tokens, "Masked prefix:", prefix_length)
print("Supervised answer:", decoded)
print("Collated shapes:", {k: tuple(v.shape) for k, v in
      collate([example, example], processor_check.tokenizer.pad_token_id).items()})
del example, supervised, processor_check


## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
def setup_model(model, cfg, run_dir):
    import torch
    from peft import LoraConfig, get_peft_model
    include_vit = cfg["setup"] == "e3"
    names = target_names([n for n, _ in model.named_modules()], include_vit)
    expected = 348 if include_vit else 252
    if len(names) != expected:
        raise RuntimeError(f"LoRA matched {len(names)} modules; expected {expected}. Check model tree.")
    for p in model.parameters():
        p.requires_grad_(False)
    model.enable_input_require_grads()
    pattern = LLM_TARGET if not include_vit else f"(?:{LLM_TARGET}|{VIT_TARGET})"
    model = get_peft_model(model, LoraConfig(
        r=cfg["rank"], lora_alpha=2 * cfg["rank"], lora_dropout=0.05,
        target_modules=pattern, bias="none", task_type="CAUSAL_LM"))
    if cfg["setup"] in ("e2", "e3"):
        for n, p in model.named_parameters():
            if any(s in n for s in PROJECTORS):
                p.requires_grad_(True)
    # GradScaler cannot unscale FP16 trainable gradients. Frozen visual remains FP16.
    for p in model.parameters():
        if p.requires_grad:
            p.data = p.data.to(torch.float32)
    trainable = [(n, p.numel()) for n, p in model.named_parameters() if p.requires_grad]
    write_json(run_dir / "trainable_parameters.json", {
        "matched_modules": names, "matched_count": len(names), "parameters": trainable,
        "trainable_params": sum(v for _, v in trainable)})
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.config.use_cache = False
    return model, sum(v for _, v in trainable)


In [ ]:
# In cấu hình và số module mục tiêu trước khi train.
modules = (ROOT / "work/e0/module_names.txt").read_text(encoding="utf-8").splitlines()
for name in ("e1", "e2", "e3"):
    cfg = config_for(base, name)
    matched = target_names(modules, include_vit=(name == "e3"))
    assert len(matched) == (348 if name == "e3" else 252)
    print(name.upper(), "LoRA modules:", len(matched), "rank:", cfg["rank"],
          "train_projector:", name in ("e2", "e3"))
print("Architecture:", json.loads((ROOT / "work/e0/architecture.json").read_text()))


## 6. Vòng huấn luyện

In [ ]:
def save_weights(model, processor, run_dir, step):
    import torch
    adapter = run_dir / "adapter"
    adapter.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(adapter, safe_serialization=True)
    processor.save_pretrained(adapter)
    extras = {n: p.detach().float().cpu() for n, p in model.named_parameters()
              if p.requires_grad and any(s in n for s in PROJECTORS)}
    if extras:
        temp = run_dir / "projector.pt.tmp"
        torch.save(extras, temp)
        temp.replace(run_dir / "projector.pt")
    write_json(run_dir / "checkpoint.json", {"optimizer_step": step,
                                             "projector_saved": bool(extras)})
    if BACKUP_ROOT is not None:
        shutil.copytree(run_dir, BACKUP_ROOT / "work" / run_dir.name, dirs_exist_ok=True)


def train(model, processor, records, cfg, run_dir, trainable_params):
    import torch
    from torch.utils.data import DataLoader
    from transformers import get_cosine_schedule_with_warmup
    start = time.perf_counter()
    torch.cuda.reset_peak_memory_stats()
    records = subset_train(records, cfg["train_fraction"], cfg["seed"])
    write_json(run_dir / "train_image_ids.json", [r["image_id"] for r in records])
    encoded = []
    for i, row in enumerate(records):
        item, answer, n = encode(row, processor)
        if i == 0:
            supervised = item["labels"][item["labels"] != -100]
            decoded = processor.tokenizer.decode(supervised, skip_special_tokens=False)
            image_tokens = int((item["input_ids"] == model.config.image_token_id).sum())
            if (row["width"], row["height"]) == (640, 360) and image_tokens != 220:
                raise RuntimeError(f"Image has {image_tokens} vision tokens; expected 220.")
            if not decoded.startswith(answer) or "<|im_end|>" not in decoded:
                raise RuntimeError("Supervised tokens do not contain the complete answer and EOS.")
            write_json(run_dir / "mask_check.json", {
                "image_id": row["image_id"], "image_tokens": image_tokens,
                "masked_prefix_tokens": n, "supervised_tokens": int(supervised.numel()),
                "answer": answer, "decoded_supervised": decoded, "prefix_equal": True})
        encoded.append(item)
        if (i + 1) % 200 == 0:
            print(f"Encoded {i + 1}/{len(records)} training images", flush=True)
    generator = torch.Generator().manual_seed(cfg["seed"])
    loader = DataLoader(encoded, batch_size=cfg["batch_size"], shuffle=True,
                        generator=generator, num_workers=0,
                        collate_fn=lambda b: collate(b, processor.tokenizer.pad_token_id))
    iterator = iter(loader)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=cfg["lr"], weight_decay=0)
    scheduler = get_cosine_schedule_with_warmup(
        opt, num_warmup_steps=max(1, round(cfg["max_steps"] * 0.05)),
        num_training_steps=cfg["max_steps"])
    scaler = torch.amp.GradScaler("cuda")
    model.train()
    opt.zero_grad(set_to_none=True)
    step = attempts = skipped = 0
    with (run_dir / "train_log.jsonl").open("w", encoding="utf-8") as log:
        while step < cfg["max_steps"]:
            attempts += 1
            loss_sum = 0.0
            for _ in range(cfg["grad_accum"]):
                try:
                    batch = next(iterator)
                except StopIteration:
                    iterator = iter(loader)
                    batch = next(iterator)
                batch = {k: v.to("cuda:0") for k, v in batch.items()}
                with torch.autocast("cuda", dtype=torch.float16):
                    raw_loss = model(**batch).loss
                    loss = raw_loss / cfg["grad_accum"]
                if not torch.isfinite(raw_loss):
                    raise RuntimeError(f"Non-finite loss at attempt {attempts}: {raw_loss.item()}")
                loss_sum += raw_loss.detach().item() / cfg["grad_accum"]
                scaler.scale(loss).backward()
                del batch, loss, raw_loss
            scaler.unscale_(opt)
            grad_norm = torch.nn.utils.clip_grad_norm_(params, 1.0)
            old_scale = scaler.get_scale()
            lr_used = opt.param_groups[0]["lr"]
            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True)
            did_step = scaler.get_scale() >= old_scale
            if did_step:
                scheduler.step()
                step += 1
                skipped = 0
            else:
                skipped += 1
            event = {"optimizer_step": step, "attempt": attempts, "loss": loss_sum,
                     "lr": lr_used, "grad_norm": float(grad_norm) if torch.isfinite(grad_norm) else None,
                     "amp_scale": scaler.get_scale(), "update_skipped": not did_step,
                     "elapsed_minutes": (time.perf_counter() - start) / 60}
            log.write(json.dumps(event) + "\n")
            log.flush()
            if skipped >= 10:
                raise RuntimeError("10 consecutive AMP overflows; inspect loss and trainable dtypes.")
            if did_step and (step == 1 or step % 10 == 0):
                print(f"{cfg['name']} step {step}/{cfg['max_steps']} loss={loss_sum:.4f} "
                      f"VRAM={torch.cuda.max_memory_allocated()/2**30:.2f} GiB", flush=True)
            if did_step and (step % 100 == 0 or step == cfg["max_steps"]):
                save_weights(model, processor, run_dir, step)
    torch.cuda.synchronize()
    meta = {"trainable_params": trainable_params,
            "peak_vram_gb": torch.cuda.max_memory_allocated() / 2**30,
            "train_minutes": (time.perf_counter() - start) / 60}
    write_json(run_dir / "run.json", {**meta, "config": cfg, "optimizer_steps": step,
                                      "attempts": attempts, "n_train_images": len(records),
                                      "status": "trained", "vram_unit": "GiB (2**30 bytes)"})
    return meta


In [ ]:
def train_setup(cfg):
    """Huấn luyện một setup độc lập. Chạy lại cell có thể dùng checkpoint train hoàn tất cùng config."""
    name = cfg["name"]
    run_dir = ROOT / "work" / name
    run_dir.mkdir(parents=True, exist_ok=True)
    model = processor = None
    try:
        if restore_completed(base, cfg):
            return
        existing = run_dir / "run.json"
        if existing.exists() and (run_dir / "adapter/adapter_config.json").exists():
            run = json.loads(existing.read_text(encoding="utf-8"))
            if (run.get("status") == "trained" and run.get("optimizer_steps") == cfg["max_steps"]
                and fingerprint(run["config"]) == fingerprint(cfg)
                and (cfg["setup"] == "e1" or (run_dir / "projector.pt").exists())):
                print("Dùng lại checkpoint train đã hoàn thành:", name)
                return
        # Xóa marker lượt cũ trước khi train lại để không tái dùng nhầm bằng chứng.
        if (run_dir / "completed.json").exists():
            (run_dir / "completed.json").unlink()
        seed_everything(cfg["seed"])
        write_json(run_dir / "effective_config.json", cfg)
        model, processor = load_base(cfg)
        write_json(run_dir / "architecture.json", architecture(model))
        model, trainable_params = setup_model(model, cfg, run_dir)
        print(name, "trainable_params:", trainable_params)
        train(model, processor, splits["train"], cfg, run_dir, trainable_params)
    except Exception as exc:
        write_json(ROOT / "report/failure.json", {
            "phase": "train", "experiment": name, "error": str(exc), "traceback": traceback.format_exc()})
        raise
    finally:
        model = processor = None
        gc.collect()
        torch.cuda.empty_cache()
        bundle_results()
        persist_outputs()



### Huấn luyện E1 độc lập từ model gốc

In [ ]:
cfg_e1 = config_for(base, "e1")
train_setup(cfg_e1)


### Huấn luyện E2 độc lập từ model gốc

In [ ]:
cfg_e2 = config_for(base, "e2")
train_setup(cfg_e2)


### Huấn luyện E3 độc lập từ model gốc

In [ ]:
cfg_e3 = config_for(base, "e3")
train_setup(cfg_e3)


## 7. Suy luận và chấm E1–E3

In [ ]:
def reload_weights(model, run_dir, cfg):
    import torch
    from peft import PeftModel
    model = PeftModel.from_pretrained(model, run_dir / "adapter", is_trainable=False)
    if cfg["setup"] in ("e2", "e3"):
        path = run_dir / "projector.pt"
        if not path.exists():
            raise RuntimeError("Trained projector missing: refusing to evaluate E2/E3.")
        extras = torch.load(path, map_location="cpu", weights_only=True)
        params = dict(model.named_parameters())
        expected = {n for n in params if any(s in n for s in PROJECTORS)}
        if set(extras) != expected:
            raise RuntimeError("Projector checkpoint names do not match the loaded model.")
        with torch.no_grad():
            for n, saved in extras.items():
                # Keep the saved FP32 projector; autocast handles linear operations.
                params[n].data = saved.to(device=params[n].device, dtype=torch.float32)
        write_json(run_dir / "projector_reload.json", {
            "verified": all(torch.equal(params[n].detach().cpu(), t) for n, t in extras.items()),
            "tensors": len(extras)})
    return model


### Nạp checkpoint và đánh giá E1

In [ ]:
evaluate_setup(cfg_e1)


### Nạp checkpoint và đánh giá E2

In [ ]:
evaluate_setup(cfg_e2)


### Nạp checkpoint và đánh giá E3

In [ ]:
evaluate_setup(cfg_e3)


In [ ]:
write_summary(ROOT / "report/evidence", ROOT / "report/summary.csv")
import pandas as pd
display(pd.read_csv(ROOT / "report/summary.csv"))


## 8. E4: ablation data_size; E5 là bonus tùy chọn

In [ ]:
# Chọn baseline chỉ bằng full validation F1; không dùng test để chọn.
vals = {n: json.loads((ROOT / f"report/validation/{n}.json").read_text(encoding="utf-8"))
        for n in ("e1", "e2", "e3")}
best = max(vals, key=lambda n: vals[n]["micro"]["f1"])
write_json(ROOT / "report/e4_selection.json", {
    "selected_setup": best, "criterion": "full validation micro F1; ties prefer E1 then E2",
    "validation_f1": {n: d["micro"]["f1"] for n, d in vals.items()},
    "variable": "data_size", "values": [0.1, 1.0], "baseline_100_percent_reuses": best,
    "test_used_for_selection": False})
baseline = json.loads((ROOT / f"report/evidence/{best}.json").read_text(encoding="utf-8"))
baseline["name"] = "E4_DATA_SIZE_100"
baseline["meta"].update(ablation_variable="data_size", train_fraction=1.0,
                        baseline_setup=best, reused_from=best)
write_json(ROOT / "report/evidence/e4_data_size_100.json", baseline)
print("E4 baseline:", best, "10% vs 100%, mọi cấu hình train khác giữ nguyên")


In [ ]:
cfg_e4 = config_for(base, "e4_data_size_10", setup=best, fraction=0.1)
train_setup(cfg_e4)


In [ ]:
evaluate_setup(cfg_e4)
evidence_path = ROOT / "report/evidence/e4_data_size_10.json"
evidence = json.loads(evidence_path.read_text(encoding="utf-8"))
evidence["meta"].update(ablation_variable="data_size", train_fraction=0.1, baseline_setup=best)
write_json(evidence_path, evidence)
bundle_results()


### Kiểm tra bằng chứng và xuất kết quả để viết báo cáo

In [ ]:
# Chấm cùng output E1 theo thang pixel sai để giải thích câu 3 của báo cáo.
command([sys.executable, "scripts/score.py", "--pred", "work/e1.jsonl", "--coord", "pixel",
         "--name", "E1_WRONG_COORD_DIAGNOSTIC", "--out", "report/diagnostics/e1_wrong_coord.json"])
e0 = json.loads((ROOT / "report/evidence/e0.json").read_text(encoding="utf-8"))
e1 = json.loads((ROOT / "report/evidence/e1.json").read_text(encoding="utf-8"))
quality = {
    "e1_parse_valid_at_least_095": e1["parse_valid_rate"] >= 0.95,
    "e1_f1_gain_at_least_010": e1["micro"]["f1"] - e0["micro"]["f1"] >= 0.10,
    "e1_f1_gain": e1["micro"]["f1"] - e0["micro"]["f1"],
    "all_test_predictions_present": all(json.loads(p.read_text())["missing_predictions"] == 0
                                        for p in (ROOT / "report/evidence").glob("*.json")),
    "status": "finished", "report_and_git_checks_pending": True,
    "smoke_run": base["max_steps"] != 300}
write_json(ROOT / "report/quality_checks.json", quality)
result = subprocess.run([sys.executable, "scripts/check_submission.py"], cwd=ROOT,
                        capture_output=True, text=True, encoding="utf-8", errors="replace")
(ROOT / "report/submission_check.txt").write_text(result.stdout + result.stderr, encoding="utf-8")
if (ROOT / "report/failure.json").exists():
    (ROOT / "report/failure.json").unlink()
bundle_results()
print(json.dumps(quality, indent=2))

persist_outputs()


In [ ]:
write_summary(ROOT / "report/evidence", ROOT / "report/summary.csv")
import pandas as pd
display(pd.read_csv(ROOT / "report/summary.csv"))
print((ROOT / "report/quality_checks.json").read_text(encoding="utf-8"))
persist_outputs()
archive = ROOT.parent / "lab_day10_results.zip"
print("Results:", archive)
print("Checkpoint:", ROOT / "work")
if DOWNLOAD_RESULTS:
    from google.colab import files
    files.download(str(archive))
print("Gửi ZIP để phân tích và hoàn thành REPORT.md bằng số liệu thật.")
